In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# STEP 4: LOAD CORE MIMIC-IV TABLES
# ============================================================

PROJECT_DIR = Path(
    r"C:\Users\sowmy\Downloads\MIMIC-IV\MIMIC_IV\MIMIC_IV"
)
# ============================================================
# FIGURE OUTPUT FOLDERS
# ============================================================

DATA_VIS_DIR = (
    PROJECT_DIR /
    "data visualizations"
)

FEATURE_VIS_DIR = (
    PROJECT_DIR /
    "feature engineering figures"
)

MODEL_VIS_DIR = (
    PROJECT_DIR /
    "model training figures"
)


# Create folders automatically if they do not exist
DATA_VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_VIS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("\nFigure folders created:")
print("Data visualizations:", DATA_VIS_DIR)
print("Feature engineering:", FEATURE_VIS_DIR)
print("Model training:", MODEL_VIS_DIR)
BASE = (
    PROJECT_DIR
    / "physionet.org"
    / "files"
    / "mimiciv"
    / "3.1"
)

HOSP = BASE / "hosp"
ICU = BASE / "icu"


# ------------------------------------------------------------
# LOAD TABLES
# ------------------------------------------------------------

print("Loading patients...")
patients = pd.read_csv(
    HOSP / "patients.csv"
)

print("Loading admissions...")
admissions = pd.read_csv(
    HOSP / "admissions.csv"
)

print("Loading ICU stays...")
icustays = pd.read_csv(
    ICU / "icustays.csv"
)

print("Loading lab dictionary...")
labitems = pd.read_csv(
    HOSP / "d_labitems.csv"
)

print("Loading diagnoses...")
diagnoses = pd.read_csv(
    HOSP / "diagnoses_icd.csv"
)
# ------------------------------------------------------------
# CHECK SHAPES
# ------------------------------------------------------------

print("\n" + "=" * 50)
print("DATASET SIZES")
print("=" * 50)

print("Patients:      ", patients.shape)
print("Admissions:    ", admissions.shape)
print("ICU stays:     ", icustays.shape)
print("Lab dictionary:", labitems.shape)
print("Diagnoses:     ", diagnoses.shape)

# ------------------------------------------------------------
# CHECK COLUMNS
# ------------------------------------------------------------

print("\nPATIENT COLUMNS:")
print(patients.columns.tolist())

print("\nADMISSION COLUMNS:")
print(admissions.columns.tolist())

print("\nICU STAY COLUMNS:")
print(icustays.columns.tolist())


# ------------------------------------------------------------
# SHOW FIRST FEW ROWS
# ------------------------------------------------------------

print("\nFirst 5 patient records:")
print(patients.head())

print("\nFirst 5 admission records:")
print(admissions.head())

print("\nFirst 5 ICU stay records:")
print(icustays.head())


print("\nSTEP 4 COMPLETE")
# ============================================================
# STEP 5: CLEAN AND VALIDATE TIMESTAMPS
# ============================================================

print("\n" + "=" * 50)
print("STEP 5: TIMESTAMP CLEANING")
print("=" * 50)


# ------------------------------------------------------------
# 1. CONVERT ADMISSION TIMESTAMPS
# ------------------------------------------------------------

admission_time_columns = [
    "admittime",
    "dischtime",
    "deathtime",
    "edregtime",
    "edouttime"
]

for col in admission_time_columns:

    admissions[col] = pd.to_datetime(
        admissions[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# 2. CONVERT ICU TIMESTAMPS
# ------------------------------------------------------------

for col in ["intime", "outtime"]:

    icustays[col] = pd.to_datetime(
        icustays[col],
        errors="coerce"
    )


# ------------------------------------------------------------
# 3. CHECK MISSING ESSENTIAL TIMESTAMPS
# ------------------------------------------------------------

print("\nMissing admission timestamps:")

print(
    admissions[
        ["admittime", "dischtime"]
    ]
    .isna()
    .sum()
)


print("\nMissing ICU timestamps:")

print(
    icustays[
        ["intime", "outtime"]
    ]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# 4. CHECK IMPOSSIBLE HOSPITAL TIMES
# ------------------------------------------------------------

invalid_admissions = (
    admissions["dischtime"]
    <
    admissions["admittime"]
)

print(
    "\nDischarge before admission:",
    invalid_admissions.sum()
)


# ------------------------------------------------------------
# 5. CHECK IMPOSSIBLE ICU TIMES
# ------------------------------------------------------------

invalid_icu = (
    icustays["outtime"]
    <
    icustays["intime"]
)

print(
    "ICU discharge before ICU admission:",
    invalid_icu.sum()
)


# ------------------------------------------------------------
# 6. REMOVE INVALID RECORDS IF PRESENT
# ------------------------------------------------------------

admissions = admissions[
    ~invalid_admissions
].copy()

icustays = icustays[
    ~invalid_icu
].copy()


# ------------------------------------------------------------
# 7. CHECK DUPLICATE PRIMARY IDS
# ------------------------------------------------------------

print(
    "\nDuplicate hadm_id in admissions:",
    admissions["hadm_id"]
    .duplicated()
    .sum()
)

print(
    "Duplicate stay_id in icustays:",
    icustays["stay_id"]
    .duplicated()
    .sum()
)


# ------------------------------------------------------------
# FINAL SIZE
# ------------------------------------------------------------

print(
    "\nAdmissions after timestamp cleaning:",
    admissions.shape
)

print(
    "ICU stays after timestamp cleaning:",
    icustays.shape
)


print("\nSTEP 5 COMPLETE")
# ============================================================
# STEP 6: BUILD ADULT PATIENT COHORT
# ============================================================

print("\n" + "=" * 50)
print("STEP 6: BUILD ADULT COHORT")
print("=" * 50)


# ------------------------------------------------------------
# 1. MERGE PATIENT INFORMATION WITH ADMISSIONS
# ------------------------------------------------------------

cohort = admissions.merge(
    patients[
        [
            "subject_id",
            "gender",
            "anchor_age",
            "anchor_year"
        ]
    ],
    on="subject_id",
    how="left"
)

print(
    "\nAdmissions after patient merge:",
    cohort.shape
)


# ------------------------------------------------------------
# 2. CALCULATE AGE AT HOSPITAL ADMISSION
# ------------------------------------------------------------

cohort["admission_year"] = (
    cohort["admittime"].dt.year
)

cohort["age"] = (
    cohort["anchor_age"]
    + cohort["admission_year"]
    - cohort["anchor_year"]
)


# ------------------------------------------------------------
# 3. INSPECT AGE BEFORE FILTERING
# ------------------------------------------------------------

print("\nAge distribution before filtering:")

print(
    cohort["age"].describe()
)


print(
    "\nMissing age:",
    cohort["age"].isna().sum()
)


# ------------------------------------------------------------
# 4. KEEP ADULT PATIENTS
# ------------------------------------------------------------

before_filter = len(cohort)

cohort = cohort[
    cohort["age"] >= 18
].copy()

after_filter = len(cohort)


print(
    "\nAdmissions removed because age < 18:",
    before_filter - after_filter
)


# ------------------------------------------------------------
# 5. CHECK FINAL COHORT
# ------------------------------------------------------------

print(
    "\nAdult admissions:",
    len(cohort)
)

print(
    "Unique adult patients:",
    cohort["subject_id"].nunique()
)

print(
    "\nAge distribution after filtering:"
)

print(
    cohort["age"].describe()
)


print("\nSTEP 6 COMPLETE")
# ============================================================
# STEP 6B: IDENTIFY KIDNEY-DISEASE COHORT
# AKI + CKD USING ICD CODES
# ============================================================

print("\n" + "=" * 50)
print("STEP 6B: IDENTIFY KIDNEY-DISEASE COHORT")
print("=" * 50)


# ------------------------------------------------------------
# 1. CLEAN ICD CODES
# ------------------------------------------------------------

diagnoses["icd_code"] = (
    diagnoses["icd_code"]
    .astype(str)
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)


# ------------------------------------------------------------
# 2. IDENTIFY ACUTE KIDNEY INJURY (AKI)
#
# ICD-9: 584.x
# ICD-10: N17.x
# ------------------------------------------------------------

aki_mask = (

    (
        (diagnoses["icd_version"] == 9)
        &
        diagnoses["icd_code"].str.startswith("584")
    )

    |

    (
        (diagnoses["icd_version"] == 10)
        &
        diagnoses["icd_code"].str.startswith("N17")
    )

)


# ------------------------------------------------------------
# 3. IDENTIFY CHRONIC KIDNEY DISEASE (CKD)
#
# ICD-9: 585.x
# ICD-10: N18.x
# ------------------------------------------------------------

ckd_mask = (

    (
        (diagnoses["icd_version"] == 9)
        &
        diagnoses["icd_code"].str.startswith("585")
    )

    |

    (
        (diagnoses["icd_version"] == 10)
        &
        diagnoses["icd_code"].str.startswith("N18")
    )

)


# ------------------------------------------------------------
# 4. KEEP AKI / CKD DIAGNOSIS RECORDS
# ------------------------------------------------------------

kidney_diagnoses = diagnoses[
    aki_mask | ckd_mask
].copy()


kidney_diagnoses["AKI"] = (
    aki_mask.loc[kidney_diagnoses.index]
    .astype(int)
)

kidney_diagnoses["CKD"] = (
    ckd_mask.loc[kidney_diagnoses.index]
    .astype(int)
)


# ------------------------------------------------------------
# 5. CREATE ONE RECORD PER HOSPITAL ADMISSION
# ------------------------------------------------------------

kidney_status = (

    kidney_diagnoses

    .groupby(
        "hadm_id",
        as_index=False
    )

    .agg(
        AKI=("AKI", "max"),
        CKD=("CKD", "max")
    )

)


# ------------------------------------------------------------
# 6. LABEL KIDNEY-DISEASE TYPE
# ------------------------------------------------------------

kidney_status["kidney_group"] = np.select(

    [
        (kidney_status["AKI"] == 1)
        &
        (kidney_status["CKD"] == 0),

        (kidney_status["AKI"] == 0)
        &
        (kidney_status["CKD"] == 1),

        (kidney_status["AKI"] == 1)
        &
        (kidney_status["CKD"] == 1)
    ],

    [
        "AKI only",
        "CKD only",
        "AKI + CKD"
    ],

    default="Unknown"
)


# ------------------------------------------------------------
# 7. SHOW KIDNEY-DISEASE COUNTS
# ------------------------------------------------------------

print("\nKidney-disease admissions:")

print(
    kidney_status[
        "kidney_group"
    ].value_counts()
)


print(
    "\nTotal kidney-disease admissions:",
    kidney_status["hadm_id"].nunique()
)


# ------------------------------------------------------------
# 8. FILTER ADULT COHORT
# ------------------------------------------------------------

before_kidney_filter = len(cohort)

cohort = cohort.merge(
    kidney_status,
    on="hadm_id",
    how="inner"
)


# ------------------------------------------------------------
# 9. VERIFY FINAL KIDNEY COHORT
# ------------------------------------------------------------

print(
    "\nAdult admissions before kidney filter:",
    before_kidney_filter
)

print(
    "Adult kidney-disease admissions:",
    len(cohort)
)

print(
    "Unique kidney-disease patients:",
    cohort["subject_id"].nunique()
)


print("\nKidney groups:")

print(
    cohort[
        "kidney_group"
    ].value_counts()
)


print("\nSTEP 6B COMPLETE")
# ============================================================
# STEP 7: IDENTIFY FIRST ICU ENTRY
# ============================================================

print("\n" + "=" * 50)
print("STEP 7: IDENTIFY FIRST ICU ENTRY")
print("=" * 50)


# ------------------------------------------------------------
# 1. SORT ICU STAYS BY ENTRY TIME
# ------------------------------------------------------------

icustays_sorted = icustays.sort_values(
    ["hadm_id", "intime"]
).copy()


# ------------------------------------------------------------
# 2. KEEP FIRST ICU STAY PER HOSPITAL ADMISSION
# ------------------------------------------------------------

first_icu = (
    icustays_sorted
    .groupby(
        "hadm_id",
        as_index=False
    )
    .first()
)


# ------------------------------------------------------------
# 3. KEEP REQUIRED ICU INFORMATION
# ------------------------------------------------------------

first_icu = first_icu[
    [
        "hadm_id",
        "stay_id",
        "first_careunit",
        "intime"
    ]
].rename(
    columns={
        "intime": "icu_intime"
    }
)


# ------------------------------------------------------------
# 4. MERGE ICU INFORMATION INTO COHORT
# ------------------------------------------------------------

cohort = cohort.merge(
    first_icu,
    on="hadm_id",
    how="left"
)


# ------------------------------------------------------------
# 5. CALCULATE HOURS FROM HOSPITAL ADMISSION TO ICU
# ------------------------------------------------------------

cohort["hours_to_icu"] = (
    cohort["icu_intime"]
    - cohort["admittime"]
).dt.total_seconds() / 3600


# ------------------------------------------------------------
# 6. CHECK RESULTS
# ------------------------------------------------------------

print(
    "\nHospital admissions with an ICU stay:",
    cohort["icu_intime"].notna().sum()
)

print(
    "Hospital admissions without an ICU stay:",
    cohort["icu_intime"].isna().sum()
)


print("\nTime from hospital admission to first ICU entry (hours):")

print(
    cohort.loc[
        cohort["icu_intime"].notna(),
        "hours_to_icu"
    ].describe()
)


# ------------------------------------------------------------
# 7. CHECK ICU ENTRIES BEFORE HOSPITAL ADMISSION
# ------------------------------------------------------------

negative_times = (
    cohort["hours_to_icu"] < 0
).sum()

print(
    "\nICU entries occurring before admittime:",
    negative_times
)


print("\nSTEP 7 COMPLETE")
# ============================================================
# STEP 8: CLEAN INVALID ICU TIMING
# ============================================================

print("\n" + "=" * 50)
print("STEP 8: CLEAN INVALID ICU TIMING")
print("=" * 50)


# ------------------------------------------------------------
# 1. IDENTIFY INVALID ICU TIMES
# ------------------------------------------------------------

invalid_icu_timing = (
    cohort["hours_to_icu"] < 0
)

print(
    "\nAdmissions with ICU entry before hospital admission:",
    invalid_icu_timing.sum()
)


# ------------------------------------------------------------
# 2. REMOVE THESE RECORDS
# ------------------------------------------------------------

before = len(cohort)

cohort = cohort[
    ~invalid_icu_timing
].copy()

after = len(cohort)


print(
    "Records removed:",
    before - after
)

print(
    "Admissions remaining:",
    after
)


# ------------------------------------------------------------
# 3. VERIFY
# ------------------------------------------------------------

print(
    "\nNegative hours_to_icu remaining:",
    (cohort["hours_to_icu"] < 0).sum()
)


print(
    "\nValid ICU admissions remaining:",
    cohort["icu_intime"].notna().sum()
)


print(
    "Admissions without ICU stay:",
    cohort["icu_intime"].isna().sum()
)


print("\nUpdated hours_to_icu distribution:")

print(
    cohort.loc[
        cohort["icu_intime"].notna(),
        "hours_to_icu"
    ].describe()
)


print("\nSTEP 8 COMPLETE")
# ============================================================
# STEP 9: DEFINE ICU TRANSFER PREDICTION COHORT
# ============================================================

print("\n" + "=" * 50)
print("STEP 9: DEFINE PREDICTION COHORT")
print("=" * 50)


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

OBSERVATION_WINDOW = 24       # first 24 h used for features
PREDICTION_HORIZON = 72      # predict ICU transfer by 72 h


# ------------------------------------------------------------
# 1. POSITIVE CASES
# ICU transfer occurs after observation window,
# but within 72 h of hospital admission
# ------------------------------------------------------------

positive_mask = (
    (cohort["hours_to_icu"] > OBSERVATION_WINDOW)
    &
    (cohort["hours_to_icu"] <= PREDICTION_HORIZON)
)


# ------------------------------------------------------------
# 2. EARLY ICU ADMISSIONS
# These cannot be used because ICU happened during
# our 24-hour feature observation period
# ------------------------------------------------------------

early_icu_mask = (
    cohort["hours_to_icu"].notna()
    &
    (cohort["hours_to_icu"] <= OBSERVATION_WINDOW)
)


# ------------------------------------------------------------
# 3. NEGATIVE CASES
#
# No ICU admission within first 72 h.
# Includes:
# - never admitted to ICU
# - ICU admission only after 72 h
# ------------------------------------------------------------

negative_mask = (
    cohort["hours_to_icu"].isna()
    |
    (cohort["hours_to_icu"] > PREDICTION_HORIZON)
)


# ------------------------------------------------------------
# 4. DISPLAY COUNTS BEFORE FILTERING
# ------------------------------------------------------------

print(
    "\nICU within first 24 h (excluded):",
    early_icu_mask.sum()
)

print(
    "ICU between 24–72 h (positive):",
    positive_mask.sum()
)

print(
    "No ICU within 72 h (negative):",
    negative_mask.sum()
)


# ------------------------------------------------------------
# 5. CREATE ELIGIBLE COHORT
# ------------------------------------------------------------

eligible_mask = (
    positive_mask
    |
    negative_mask
)

model_cohort = cohort[
    eligible_mask
].copy()


# ------------------------------------------------------------
# 6. CREATE TARGET
# ------------------------------------------------------------

model_cohort["icu_transfer"] = (
    (
        model_cohort["hours_to_icu"] > 24
    )
    &
    (
        model_cohort["hours_to_icu"] <= 72
    )
).astype(int)


# ------------------------------------------------------------
# 7. CHECK TARGET DISTRIBUTION
# ------------------------------------------------------------

print("\nFinal prediction cohort:")

print(
    model_cohort[
        "icu_transfer"
    ].value_counts()
)


print("\nPercentages:")

print(
    model_cohort[
        "icu_transfer"
    ]
    .value_counts(
        normalize=True
    )
    * 100
)


print(
    "\nTotal eligible admissions:",
    len(model_cohort)
)

print(
    "Unique patients:",
    model_cohort[
        "subject_id"
    ].nunique()
)


# ------------------------------------------------------------
# 8. VERIFY NO EARLY ICU CASES REMAIN
# ------------------------------------------------------------

remaining_early_icu = (
    model_cohort["hours_to_icu"].notna()
    &
    (
        model_cohort["hours_to_icu"]
        <= 24
    )
).sum()


print(
    "\nEarly ICU cases remaining:",
    remaining_early_icu
)


print("\nSTEP 9 COMPLETE")
# ============================================================
# STEP 10: SELECT CANDIDATE LABORATORY FEATURES
# ============================================================

print("\n" + "=" * 50)
print("STEP 10: SELECT CANDIDATE LABS")
print("=" * 50)


# ------------------------------------------------------------
# LABS WE WANT TO LOOK FOR
# ------------------------------------------------------------

lab_search_terms = [
    # Renal
    "Creatinine",
    "Urea Nitrogen",

    # Electrolytes
    "Sodium",
    "Potassium",
    "Chloride",
    "Bicarbonate",
    "Calcium",
    "Magnesium",
    "Phosphate",

    # Metabolic / perfusion
    "Glucose",
    "Lactate",
    "Anion Gap",

    # Hematology
    "White Blood Cells",
    "Hemoglobin",
    "Hematocrit",
    "Platelet",
    "RDW",

    # Liver
    "Bilirubin",
    "Albumin",
    "Alanine Aminotransferase",
    "Asparate Aminotransferase",
    "Alkaline Phosphatase",

    # Coagulation
    "INR",
    "PT",
    "PTT",

    # Blood gas
    "pH",
    "pCO2",
    "pO2"
]


# ------------------------------------------------------------
# SEARCH THE LAB DICTIONARY
# ------------------------------------------------------------

matches = []

for term in lab_search_terms:

    result = labitems[
        labitems["label"]
        .str.contains(
            term,
            case=False,
            na=False,
            regex=False
        )
    ].copy()

    if len(result) > 0:

        result["search_term"] = term

        matches.append(result)


candidate_labitems = pd.concat(
    matches,
    ignore_index=True
)


# ------------------------------------------------------------
# REMOVE DUPLICATE ITEM IDs
# ------------------------------------------------------------

candidate_labitems = (
    candidate_labitems
    .drop_duplicates(
        subset=["itemid"]
    )
    .sort_values(
        ["search_term", "label"]
    )
)


# ------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------

print(
    "\nNumber of candidate item IDs:",
    candidate_labitems["itemid"].nunique()
)


print(
    "\nCandidate laboratory tests:\n"
)


print(
    candidate_labitems[
        [
            "itemid",
            "label",
            "fluid",
            "category",
            "search_term"
        ]
    ].to_string(
        index=False
    )
)


print("\nSTEP 10 COMPLETE")
# ============================================================
# STEP 11: DEFINE EXACT LAB FEATURES
# ============================================================

print("\n" + "=" * 50)
print("STEP 11: DEFINE EXACT LAB FEATURES")
print("=" * 50)


# ------------------------------------------------------------
# EXACT MIMIC-IV ITEM IDs
# ------------------------------------------------------------

LAB_MAP = {

    # Renal
    50912: "creatinine",
    51006: "bun",

    # Electrolytes
    50983: "sodium",
    50971: "potassium",
    50902: "chloride",
    50882: "bicarbonate",
    50893: "calcium",
    50960: "magnesium",
    50970: "phosphate",

    # Metabolic / perfusion
    50931: "glucose",
    50813: "lactate",
    50868: "anion_gap",

    # Hematology
    51301: "wbc",
    51222: "hemoglobin",
    51221: "hematocrit",
    51265: "platelets",
    51277: "rdw",

    # Liver
    50885: "bilirubin_total",
    50862: "albumin",
    50861: "alt",
    50878: "ast",
    50863: "alkaline_phosphatase",

    # Coagulation
    51237: "inr",
    51274: "pt",
    51275: "ptt",

    # Blood gas
    50820: "ph",
    50818: "pco2",
    50821: "po2"
}


# ------------------------------------------------------------
# CHECK THAT EVERY ID EXISTS
# ------------------------------------------------------------

selected_labs = labitems[
    labitems["itemid"].isin(
        LAB_MAP.keys()
    )
].copy()


selected_labs["model_feature"] = (
    selected_labs["itemid"]
    .map(LAB_MAP)
)


# Sort in model-feature order
selected_labs = selected_labs[
    [
        "itemid",
        "model_feature",
        "label",
        "fluid",
        "category"
    ]
].sort_values(
    "model_feature"
)


print(
    "\nNumber of selected laboratory variables:",
    len(LAB_MAP)
)


print(
    "\nNumber of IDs successfully found:",
    len(selected_labs)
)


print("\nSelected labs:\n")

print(
    selected_labs.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# CHECK FOR MISSING IDs
# ------------------------------------------------------------

found_ids = set(
    selected_labs["itemid"]
)

missing_ids = (
    set(LAB_MAP.keys())
    -
    found_ids
)


print(
    "\nMissing item IDs:",
    missing_ids
)


print("\nSTEP 11 COMPLETE")
# ============================================================
# STEP 12: EXTRACT REQUIRED LAB EVENTS
# ============================================================

print("\n" + "=" * 50)
print("STEP 12: EXTRACT LAB EVENTS")
print("=" * 50)

LAB_CACHE = PROJECT_DIR / "kidney_selected_labs.parquet"

# ------------------------------------------------------------
# IF CACHE EXISTS -> LOAD IT
# ------------------------------------------------------------

if LAB_CACHE.exists():

    print("\nCached lab file found.")
    print("Loading labs from cache...")

    labs = pd.read_parquet(
        LAB_CACHE
    )

    print(
        "Loaded lab measurements:",
        f"{len(labs):,}"
    )

# ------------------------------------------------------------
# OTHERWISE -> EXTRACT FROM ORIGINAL FILE
# ------------------------------------------------------------

else:

    print("\nNo cache found.")
    print("Extracting from labevents.csv...")

    LAB_IDS = set(
        LAB_MAP.keys()
    )

    ELIGIBLE_HADM_IDS = set(
        model_cohort["hadm_id"]
        .dropna()
        .astype(int)
    )

    lab_chunks = []

    CHUNK_SIZE = 1_000_000

    chunk_number = 0
    total_selected = 0

    for chunk in pd.read_csv(

        HOSP / "labevents.csv",

        usecols=[
            "subject_id",
            "hadm_id",
            "itemid",
            "charttime",
            "valuenum",
            "valueuom"
        ],

        chunksize=CHUNK_SIZE
    ):

        chunk_number += 1

        # Keep only selected labs
        chunk = chunk[
            chunk["itemid"].isin(
                LAB_IDS
            )
        ]

        # Keep only eligible admissions
        chunk = chunk[
            chunk["hadm_id"].isin(
                ELIGIBLE_HADM_IDS
            )
        ]

        # Keep numeric results
        chunk = chunk[
            chunk["valuenum"].notna()
        ]

        if not chunk.empty:

            lab_chunks.append(
                chunk
            )

            total_selected += len(
                chunk
            )

        print(
            f"Chunk {chunk_number}: "
            f"total retained = "
            f"{total_selected:,}"
        )

    labs = pd.concat(
        lab_chunks,
        ignore_index=True
    )

    del lab_chunks

    print(
        "\nExtraction finished."
    )

    print(
        "Total retained:",
        f"{len(labs):,}"
    )

    # SAVE CACHE
    print(
        "\nSaving extracted labs..."
    )

    labs.to_parquet(
        LAB_CACHE,
        index=False
    )

    print(
        "Saved successfully:"
    )

    print(
        LAB_CACHE
    )


print("\nSTEP 12 COMPLETE")
# ============================================================
# STEP 13: CLEAN LAB MEASUREMENTS
# ============================================================

print("\n" + "=" * 50)
print("STEP 13: CLEAN LAB MEASUREMENTS")
print("=" * 50)


# ------------------------------------------------------------
# 1. CONVERT LAB TIMESTAMPS
# ------------------------------------------------------------

labs["charttime"] = pd.to_datetime(
    labs["charttime"],
    errors="coerce"
)

print(
    "\nMissing charttime:",
    labs["charttime"].isna().sum()
)


# ------------------------------------------------------------
# 2. REMOVE RECORDS WITH MISSING TIMESTAMPS
# ------------------------------------------------------------

before = len(labs)

labs = labs[
    labs["charttime"].notna()
].copy()

print(
    "Removed due to missing charttime:",
    before - len(labs)
)


# ------------------------------------------------------------
# 3. REMOVE NON-FINITE NUMERIC VALUES
# ------------------------------------------------------------

before = len(labs)

labs = labs[
    np.isfinite(
        labs["valuenum"]
    )
].copy()

print(
    "Removed non-finite values:",
    before - len(labs)
)


# ------------------------------------------------------------
# 4. IDENTIFY EXACT DUPLICATES
# ------------------------------------------------------------

duplicate_mask = labs.duplicated(
    subset=[
        "subject_id",
        "hadm_id",
        "itemid",
        "charttime",
        "valuenum",
        "valueuom"
    ]
)

print(
    "\nExact duplicate measurements:",
    duplicate_mask.sum()
)


# ------------------------------------------------------------
# 5. REMOVE EXACT DUPLICATES
# ------------------------------------------------------------

before = len(labs)

labs = labs[
    ~duplicate_mask
].copy()

print(
    "Duplicates removed:",
    before - len(labs)
)


# ------------------------------------------------------------
# 6. ADD READABLE LAB NAMES
# ------------------------------------------------------------

labs["lab_name"] = (
    labs["itemid"]
    .map(LAB_MAP)
)


# ------------------------------------------------------------
# 7. FINAL CHECK
# ------------------------------------------------------------

print(
    "\nMeasurements remaining:",
    f"{len(labs):,}"
)

print(
    "Unique laboratory variables:",
    labs["lab_name"].nunique()
)

print(
    "Missing lab names:",
    labs["lab_name"].isna().sum()
)


print("\nLab measurement counts:")

print(
    labs["lab_name"]
    .value_counts()
)


print("\nSTEP 13 COMPLETE")
# ============================================================
# STEP 14: CHECK LAB MEASUREMENT UNITS
# ============================================================

print("\n" + "=" * 50)
print("STEP 14: CHECK LAB MEASUREMENT UNITS")
print("=" * 50)


# Count each unit used for each laboratory variable
unit_counts = (
    labs
    .groupby(
        ["lab_name", "valueuom"],
        dropna=False
    )
    .size()
    .reset_index(name="count")
)


# Calculate percentage within each lab
unit_counts["percent"] = (
    unit_counts["count"]
    /
    unit_counts.groupby("lab_name")["count"].transform("sum")
    * 100
)


# Sort
unit_counts = unit_counts.sort_values(
    ["lab_name", "count"],
    ascending=[True, False]
)


print("\nUnits used for each laboratory variable:\n")

print(
    unit_counts.to_string(index=False)
)


# ------------------------------------------------------------
# COUNT HOW MANY DIFFERENT UNITS EACH LAB HAS
# ------------------------------------------------------------

units_per_lab = (
    labs
    .groupby("lab_name")["valueuom"]
    .nunique(dropna=False)
    .sort_values(ascending=False)
)


print("\nNumber of units per lab:\n")

print(units_per_lab)


# ------------------------------------------------------------
# FLAG LABS WITH MULTIPLE UNITS
# ------------------------------------------------------------

multiple_units = units_per_lab[
    units_per_lab > 1
]


print("\nLabs with multiple recorded units:")

print(multiple_units)


print("\nSTEP 14 COMPLETE")
# ============================================================
# STEP 15: INSPECT LAB VALUE OUTLIERS
# ============================================================

print("\n" + "=" * 50)
print("STEP 15: INSPECT LAB VALUE OUTLIERS")
print("=" * 50)


# ------------------------------------------------------------
# CALCULATE DISTRIBUTION FOR EACH LAB
# ------------------------------------------------------------

lab_summary = (
    labs
    .groupby("lab_name")["valuenum"]
    .agg(
        count="count",
        minimum="min",
        maximum="max",
        mean="mean",
        median="median"
    )
)


# ------------------------------------------------------------
# ADD PERCENTILES
# ------------------------------------------------------------

percentiles = (
    labs
    .groupby("lab_name")["valuenum"]
    .quantile(
        [
            0.001,
            0.01,
            0.99,
            0.999
        ]
    )
    .unstack()
)


percentiles.columns = [
    "p0.1",
    "p1",
    "p99",
    "p99.9"
]


lab_summary = lab_summary.join(
    percentiles
)


# ------------------------------------------------------------
# REORDER COLUMNS
# ------------------------------------------------------------

lab_summary = lab_summary[
    [
        "count",
        "minimum",
        "p0.1",
        "p1",
        "median",
        "mean",
        "p99",
        "p99.9",
        "maximum"
    ]
]


print("\nLab value distributions:\n")

print(
    lab_summary.to_string()
)


print("\nSTEP 15 COMPLETE")
# ============================================================
# STEP 16: REMOVE PHYSIOLOGICALLY IMPLAUSIBLE LAB VALUES
# ============================================================

print("\n" + "=" * 50)
print("STEP 16: REMOVE IMPLAUSIBLE LAB VALUES")
print("=" * 50)


# ------------------------------------------------------------
# BROAD PLAUSIBILITY LIMITS
# These are intentionally wide.
# ------------------------------------------------------------

PLAUSIBLE_RANGES = {

    "albumin": (0.5, 10),
    "alkaline_phosphatase": (1, 10000),
    "alt": (1, 30000),
    "anion_gap": (0, 60),
    "ast": (1, 40000),

    "bicarbonate": (3, 60),
    "bilirubin_total": (0.1, 100),
    "bun": (1, 300),

    "calcium": (2, 20),
    "chloride": (50, 150),
    "creatinine": (0.1, 30),

    "glucose": (10, 2000),

    "hematocrit": (5, 75),
    "hemoglobin": (2, 25),

    "inr": (0.5, 30),
    "lactate": (0.1, 40),

    "magnesium": (0.2, 10),

    "pco2": (5, 200),
    "ph": (6.5, 8.0),
    "phosphate": (0.2, 20),

    "platelets": (1, 2000),

    "po2": (5, 700),

    "potassium": (1.5, 10),

    "pt": (5, 150),
    "ptt": (10, 200),

    "rdw": (5, 50),

    "sodium": (80, 180),

    "wbc": (0.1, 500)
}


# ------------------------------------------------------------
# IDENTIFY VALUES OUTSIDE PLAUSIBLE RANGE
# ------------------------------------------------------------

invalid_masks = []

for lab_name, (lower, upper) in PLAUSIBLE_RANGES.items():

    mask = (
        (labs["lab_name"] == lab_name)
        &
        (
            (labs["valuenum"] < lower)
            |
            (labs["valuenum"] > upper)
        )
    )

    count = mask.sum()

    invalid_masks.append(mask)

    print(
        f"{lab_name:25s}: "
        f"{count:,} implausible values"
    )


# ------------------------------------------------------------
# COMBINE ALL INVALID MASKS
# ------------------------------------------------------------

invalid_lab_values = np.logical_or.reduce(
    invalid_masks
)


print(
    "\nTotal implausible measurements:",
    f"{invalid_lab_values.sum():,}"
)


# ------------------------------------------------------------
# REMOVE THEM
# ------------------------------------------------------------

before = len(labs)

labs = labs[
    ~invalid_lab_values
].copy()


print(
    "Measurements removed:",
    f"{before - len(labs):,}"
)

print(
    "Measurements remaining:",
    f"{len(labs):,}"
)


# ------------------------------------------------------------
# VERIFY ALL 28 LABS REMAIN
# ------------------------------------------------------------

print(
    "\nLaboratory variables remaining:",
    labs["lab_name"].nunique()
)


print("\nSTEP 16 COMPLETE")
# ============================================================
# STEP 17: TEMPORAL FILTERING OF LABS
# ============================================================

print("\n" + "=" * 50)
print("STEP 17: TEMPORAL FILTERING")
print("=" * 50)


# ------------------------------------------------------------
# 1. GET ADMISSION TIMES FOR OUR MODEL COHORT
# ------------------------------------------------------------

admission_times = model_cohort[
    [
        "hadm_id",
        "admittime"
    ]
].copy()


# ------------------------------------------------------------
# 2. MERGE ADMISSION TIME INTO LAB DATA
# ------------------------------------------------------------

labs = labs.merge(
    admission_times,
    on="hadm_id",
    how="inner"
)


# ------------------------------------------------------------
# 3. CALCULATE HOURS SINCE HOSPITAL ADMISSION
# ------------------------------------------------------------

labs["hours_from_admission"] = (
    labs["charttime"]
    - labs["admittime"]
).dt.total_seconds() / 3600


# ------------------------------------------------------------
# 4. INSPECT TIMING BEFORE FILTERING
# ------------------------------------------------------------

print("\nLab timing before filtering:")

print(
    labs["hours_from_admission"]
    .describe()
)


# ------------------------------------------------------------
# 5. KEEP ONLY FIRST 24 HOURS
# ------------------------------------------------------------

before = len(labs)

labs_24h = labs[
    (labs["hours_from_admission"] >= 0)
    &
    (labs["hours_from_admission"] <= 24)
].copy()


print(
    "\nMeasurements before temporal filtering:",
    f"{before:,}"
)

print(
    "Measurements within first 24 h:",
    f"{len(labs_24h):,}"
)

print(
    "Measurements removed:",
    f"{before - len(labs_24h):,}"
)


# ------------------------------------------------------------
# 6. CHECK COVERAGE
# ------------------------------------------------------------

print(
    "\nAdmissions with at least one lab in first 24 h:",
    labs_24h["hadm_id"].nunique()
)

print(
    "Patients with at least one lab in first 24 h:",
    labs_24h["subject_id"].nunique()
)


# ------------------------------------------------------------
# 7. VERIFY NO FUTURE LABS REMAIN
# ------------------------------------------------------------

print(
    "\nMinimum hours from admission:",
    labs_24h["hours_from_admission"].min()
)

print(
    "Maximum hours from admission:",
    labs_24h["hours_from_admission"].max()
)


print("\nSTEP 17 COMPLETE")
# ============================================================
# STEP 18: CREATE STATIC + DYNAMIC LAB FEATURES
# ============================================================

print("\n" + "=" * 50)
print("STEP 18: LAB FEATURE ENGINEERING")
print("=" * 50)


# ------------------------------------------------------------
# 1. SORT MEASUREMENTS CHRONOLOGICALLY
# ------------------------------------------------------------

labs_24h = labs_24h.sort_values(
    [
        "hadm_id",
        "lab_name",
        "charttime"
    ]
)


# ------------------------------------------------------------
# 2. AGGREGATE EACH LAB FOR EACH ADMISSION
# ------------------------------------------------------------

lab_features = (
    labs_24h
    .groupby(
        ["hadm_id", "lab_name"]
    )
    .agg(

        first_value=(
            "valuenum",
            "first"
        ),

        latest_value=(
            "valuenum",
            "last"
        ),

        mean_value=(
            "valuenum",
            "mean"
        ),

        minimum=(
            "valuenum",
            "min"
        ),

        maximum=(
            "valuenum",
            "max"
        ),

        std=(
            "valuenum",
            "std"
        ),

        count=(
            "valuenum",
            "count"
        ),

        first_time=(
            "hours_from_admission",
            "first"
        ),

        latest_time=(
            "hours_from_admission",
            "last"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# 3. CALCULATE CHANGE
# ------------------------------------------------------------

lab_features["change"] = (
    lab_features["latest_value"]
    -
    lab_features["first_value"]
)


# ------------------------------------------------------------
# 4. CALCULATE TIME DIFFERENCE
# ------------------------------------------------------------

lab_features["time_difference"] = (
    lab_features["latest_time"]
    -
    lab_features["first_time"]
)


# ------------------------------------------------------------
# 5. CALCULATE SLOPE
#
# Only calculate when there are at least two measurements
# at different times.
# ------------------------------------------------------------

lab_features["slope"] = np.where(

    lab_features["time_difference"] > 0,

    lab_features["change"]
    /
    lab_features["time_difference"],

    np.nan
)


# ------------------------------------------------------------
# 6. CHECK OUTPUT
# ------------------------------------------------------------

print(
    "\nAdmission-lab combinations:",
    f"{len(lab_features):,}"
)

print(
    "Unique admissions:",
    lab_features["hadm_id"].nunique()
)

print(
    "Unique laboratory variables:",
    lab_features["lab_name"].nunique()
)


print("\nExample engineered features:")

print(
    lab_features.head(20).to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 7. CHECK HOW MANY HAVE TREND INFORMATION
# ------------------------------------------------------------

print(
    "\nRows with calculable slope:",
    lab_features["slope"].notna().sum()
)

print(
    "Rows without calculable slope:",
    lab_features["slope"].isna().sum()
)


print("\nSTEP 18 COMPLETE")
# ============================================================
# STEP 19: CONVERT LAB FEATURES TO WIDE FORMAT
# ============================================================

print("\n" + "=" * 50)
print("STEP 19: CREATE WIDE LAB FEATURE TABLE")
print("=" * 50)


# ------------------------------------------------------------
# FEATURES TO KEEP
# ------------------------------------------------------------

feature_types = [
    "first_value",
    "latest_value",
    "mean_value",
    "minimum",
    "maximum",
    "std",
    "count",
    "change",
    "slope"
]


# ------------------------------------------------------------
# PIVOT TO ONE ROW PER HOSPITAL ADMISSION
# ------------------------------------------------------------

lab_wide = lab_features.pivot(
    index="hadm_id",
    columns="lab_name",
    values=feature_types
)


# ------------------------------------------------------------
# FLATTEN COLUMN NAMES
#
# Example:
# ('latest_value', 'creatinine')
# becomes:
# creatinine_latest_value
# ------------------------------------------------------------

lab_wide.columns = [
    f"{lab}_{feature}"
    for feature, lab in lab_wide.columns
]


# Convert hadm_id back into a normal column
lab_wide = lab_wide.reset_index()


# ------------------------------------------------------------
# CHECK RESULT
# ------------------------------------------------------------

print(
    "\nWide dataset shape:",
    lab_wide.shape
)

print(
    "Number of admissions:",
    len(lab_wide)
)

print(
    "Number of engineered lab features:",
    len(lab_wide.columns) - 1
)


print("\nFirst 30 feature names:")

for feature in lab_wide.columns[1:31]:
    print(feature)


print(
    "\nDuplicate hadm_id:",
    lab_wide["hadm_id"].duplicated().sum()
)


print("\nSTEP 19 COMPLETE")
# ============================================================
# STEP 20: CREATE COMPLETE CANDIDATE FEATURE DATASET
# ============================================================

print("\n" + "=" * 50)
print("STEP 20: MERGE LAB + PATIENT FEATURES")
print("=" * 50)


# ------------------------------------------------------------
# 1. CREATE BASIC PATIENT / ADMISSION FEATURES
# ------------------------------------------------------------

model_cohort["admission_hour"] = (
    model_cohort["admittime"].dt.hour
)

model_cohort["admission_day"] = (
    model_cohort["admittime"].dt.dayofweek
)

model_cohort["weekend_admission"] = (
    model_cohort["admission_day"] >= 5
).astype(int)


# ------------------------------------------------------------
# 2. SELECT BASE FEATURES
# ------------------------------------------------------------

base_features = model_cohort[
    [
        # IDs - linking/splitting only
        "subject_id",
        "hadm_id",

        # Demographics
        "age",
        "gender",


        # Admission information
        "admission_type",
        "admission_location",
        "admission_hour",
        "admission_day",
        "weekend_admission",

        # Target
        "icu_transfer"
    ]
].copy()

# ------------------------------------------------------------
# 3. MERGE WITH LAB FEATURES
# ------------------------------------------------------------

model_data = base_features.merge(
    lab_wide,
    on="hadm_id",
    how="left"
)


# ------------------------------------------------------------
# 4. CHECK FINAL CANDIDATE DATASET
# ------------------------------------------------------------

print(
    "\nDataset shape:",
    model_data.shape
)

print(
    "Number of admissions:",
    len(model_data)
)

print(
    "Unique patients:",
    model_data["subject_id"].nunique()
)

print(
    "Duplicate admissions:",
    model_data["hadm_id"].duplicated().sum()
)


# ------------------------------------------------------------
# 5. CHECK TARGET
# ------------------------------------------------------------

print("\nTarget distribution:")

print(
    model_data["icu_transfer"]
    .value_counts()
)

print("\nTarget percentage:")

print(
    model_data["icu_transfer"]
    .value_counts(normalize=True)
    * 100
)


# ------------------------------------------------------------
# 6. CHECK ADMISSIONS WITHOUT ANY LAB DATA
# ------------------------------------------------------------

lab_columns = [
    col
    for col in model_data.columns
    if col in lab_wide.columns
    and col != "hadm_id"
]

no_lab_data = (
    model_data[lab_columns]
    .isna()
    .all(axis=1)
)

print(
    "\nAdmissions with NO selected labs in first 24 h:",
    no_lab_data.sum()
)

print(
    "Admissions with at least one selected lab:",
    (~no_lab_data).sum()
)


print("\nSTEP 20 COMPLETE")
# ============================================================
# STEP 21: MISSINGNESS ANALYSIS
# ============================================================

print("\n" + "=" * 50)
print("STEP 21: MISSINGNESS ANALYSIS")
print("=" * 50)


# ------------------------------------------------------------
# 1. EXCLUDE IDs AND TARGET FROM FEATURE MISSINGNESS
# ------------------------------------------------------------

non_feature_columns = [
    "subject_id",
    "hadm_id",
    "icu_transfer"
]

feature_columns = [
    col for col in model_data.columns
    if col not in non_feature_columns
]


# ------------------------------------------------------------
# 2. CALCULATE MISSINGNESS
# ------------------------------------------------------------

missingness = (
    model_data[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

missingness_df = pd.DataFrame({
    "feature": missingness.index,
    "missing_percent": missingness.values
})


# ------------------------------------------------------------
# 3. DISPLAY HIGHEST MISSINGNESS
# ------------------------------------------------------------

print("\nTop 50 features with highest missingness:\n")

print(
    missingness_df
    .head(50)
    .to_string(index=False)
)


# ------------------------------------------------------------
# 4. COUNT FEATURES BY MISSINGNESS RANGE
# ------------------------------------------------------------

print("\nMissingness categories:")

print(
    "0-20% missing:",
    (missingness <= 20).sum()
)

print(
    "20-50% missing:",
    (
        (missingness > 20)
        &
        (missingness <= 50)
    ).sum()
)

print(
    "50-80% missing:",
    (
        (missingness > 50)
        &
        (missingness <= 80)
    ).sum()
)

print(
    ">80% missing:",
    (missingness > 80).sum()
)


# ------------------------------------------------------------
# 5. FEATURES >50% MISSING
# ------------------------------------------------------------

high_missing = missingness[
    missingness > 50
]

print(
    "\nNumber of features >50% missing:",
    len(high_missing)
)

print("\nFeatures >50% missing:\n")

print(high_missing.to_string())


# ------------------------------------------------------------
# 6. SAVE REPORT
# ------------------------------------------------------------

missingness_df.to_csv(
    PROJECT_DIR / "kidney_missingness_report.csv",
    index=False
)

print(
    "\nMissingness report saved to:"
)

print(
    PROJECT_DIR / "kidney_missingness_report.csv"
)


print("\nSTEP 21 COMPLETE")
# ============================================================
# STEP 22: REMOVE REDUNDANT ENGINEERED LAB FEATURES
# ============================================================

print("\n" + "=" * 50)
print("STEP 22: REMOVE REDUNDANT LAB FEATURES")
print("=" * 50)


# ------------------------------------------------------------
# FEATURE TYPES WE WANT TO KEEP
# ------------------------------------------------------------

KEEP_SUFFIXES = (
    "_latest_value",
    "_change",
    "_slope",
    "_count"
)


# ------------------------------------------------------------
# BASE FEATURES WE ALWAYS KEEP
# ------------------------------------------------------------

BASE_FEATURES = [
    "subject_id",
    "hadm_id",

    # Demographics
    "age",
    "gender",


    # Admission information
    "admission_type",
    "admission_location",
    "admission_hour",
    "admission_day",
    "weekend_admission",

    # Target
    "icu_transfer"
]



# ------------------------------------------------------------
# SELECT LAB FEATURES
# ------------------------------------------------------------

selected_lab_features = [
    col
    for col in model_data.columns
    if col.endswith(KEEP_SUFFIXES)
]


# ------------------------------------------------------------
# CREATE REDUCED DATASET
# ------------------------------------------------------------

reduced_data = model_data[
    BASE_FEATURES
    + selected_lab_features
].copy()


# ------------------------------------------------------------
# REPORT
# ------------------------------------------------------------

print(
    "\nOriginal dataset columns:",
    model_data.shape[1]
)

print(
    "Selected lab features:",
    len(selected_lab_features)
)

print(
    "Reduced dataset columns:",
    reduced_data.shape[1]
)


print("\nFeatures retained per lab:")

print(
    "Latest value"
)

print(
    "Change"
)

print(
    "Slope"
)

print(
    "Measurement count"
)


print(
    "\nReduced dataset shape:",
    reduced_data.shape
)


print("\nSTEP 22 COMPLETE")
# ============================================================
# STEP 23: RECALCULATE MISSINGNESS AFTER FEATURE REDUCTION
# ============================================================

print("\n" + "=" * 50)
print("STEP 23: MISSINGNESS AFTER FEATURE REDUCTION")
print("=" * 50)


# ------------------------------------------------------------
# EXCLUDE IDS AND TARGET
# ------------------------------------------------------------

non_feature_columns = [
    "subject_id",
    "hadm_id",
    "icu_transfer"
]

feature_columns = [
    col
    for col in reduced_data.columns
    if col not in non_feature_columns
]


# ------------------------------------------------------------
# CALCULATE MISSINGNESS
# ------------------------------------------------------------

missingness_reduced = (
    reduced_data[feature_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)


# ------------------------------------------------------------
# DISPLAY MISSINGNESS CATEGORIES
# ------------------------------------------------------------

print("\nMissingness categories:")

print(
    "0-20% missing:",
    (missingness_reduced <= 20).sum()
)

print(
    "20-50% missing:",
    (
        (missingness_reduced > 20)
        &
        (missingness_reduced <= 50)
    ).sum()
)

print(
    "50-80% missing:",
    (
        (missingness_reduced > 50)
        &
        (missingness_reduced <= 80)
    ).sum()
)

print(
    ">80% missing:",
    (missingness_reduced > 80).sum()
)


# ------------------------------------------------------------
# PRINT ALL FEATURES AND THEIR MISSINGNESS
# ------------------------------------------------------------

print("\nFeature missingness:\n")

print(
    missingness_reduced.to_string()
)


# ------------------------------------------------------------
# SPECIFICALLY CHECK LATEST VALUES
# ------------------------------------------------------------

latest_features = [
    col
    for col in reduced_data.columns
    if col.endswith("_latest_value")
]

latest_missingness = (
    reduced_data[latest_features]
    .isna()
    .mean()
    .mul(100)
    .sort_values()
)


print("\n" + "-" * 50)
print("MISSINGNESS OF LATEST LAB VALUES")
print("-" * 50)

print(
    latest_missingness.to_string()
)


print("\nSTEP 23 COMPLETE")
# ============================================================
# STEP 24: REMOVE FEATURES WITH >50% MISSINGNESS
# ============================================================

print("\n" + "=" * 50)
print("STEP 24: REMOVE HIGH-MISSINGNESS FEATURES")
print("=" * 50)

MISSINGNESS_THRESHOLD = 50.0


# ------------------------------------------------------------
# 1. FEATURES TO REMOVE
# ------------------------------------------------------------

features_to_remove = (
    missingness_reduced[
        missingness_reduced > MISSINGNESS_THRESHOLD
    ]
    .index
    .tolist()
)


print(
    "\nFeatures removed (>50% missing):",
    len(features_to_remove)
)


# ------------------------------------------------------------
# 2. CREATE CLEANER DATASET
# ------------------------------------------------------------

clean_data = reduced_data.drop(
    columns=features_to_remove
).copy()


# ------------------------------------------------------------
# 3. REPORT
# ------------------------------------------------------------

print(
    "Columns before:",
    reduced_data.shape[1]
)

print(
    "Columns after:",
    clean_data.shape[1]
)

print(
    "Admissions:",
    clean_data.shape[0]
)


# ------------------------------------------------------------
# 4. CHECK REMAINING PREDICTOR COUNT
# ------------------------------------------------------------

non_predictors = [
    "subject_id",
    "hadm_id",
    "icu_transfer"
]

remaining_predictors = [
    col for col in clean_data.columns
    if col not in non_predictors
]


print(
    "\nRemaining candidate predictors:",
    len(remaining_predictors)
)


# ------------------------------------------------------------
# 5. VERIFY MAXIMUM MISSINGNESS
# ------------------------------------------------------------

remaining_missingness = (
    clean_data[remaining_predictors]
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)


print(
    "\nMaximum remaining missingness:",
    remaining_missingness.max()
)


print("\nRemaining predictors:\n")

for feature in remaining_predictors:
    print(feature)


print("\nSTEP 24 COMPLETE")
# ============================================================
# STEP 25: PATIENT-LEVEL TRAIN / TEST SPLIT
# ============================================================

print("\n" + "=" * 50)
print("STEP 25: PATIENT-LEVEL TRAIN / TEST SPLIT")
print("=" * 50)

from sklearn.model_selection import GroupShuffleSplit


# ------------------------------------------------------------
# 1. DEFINE FEATURES AND TARGET
# ------------------------------------------------------------

ID_COLUMNS = [
    "subject_id",
    "hadm_id"
]

TARGET = "icu_transfer"

FEATURE_COLUMNS = [
    col
    for col in clean_data.columns
    if col not in ID_COLUMNS + [TARGET]
]


X = clean_data[
    FEATURE_COLUMNS
].copy()

y = clean_data[
    TARGET
].copy()

groups = clean_data[
    "subject_id"
].copy()


# ------------------------------------------------------------
# 2. PATIENT-LEVEL SPLIT
#
# 75% training
# 25% testing
# ------------------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)


train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)


X_train = X.iloc[
    train_idx
].copy()

X_test = X.iloc[
    test_idx
].copy()

y_train = y.iloc[
    train_idx
].copy()

y_test = y.iloc[
    test_idx
].copy()


# Keep patient IDs separately for verification
train_subjects = groups.iloc[
    train_idx
]

test_subjects = groups.iloc[
    test_idx
]


# ------------------------------------------------------------
# 3. VERIFY NO PATIENT LEAKAGE
# ------------------------------------------------------------

overlap = (
    set(train_subjects)
    &
    set(test_subjects)
)


print(
    "\nPatients appearing in BOTH train and test:",
    len(overlap)
)


# ------------------------------------------------------------
# 4. DATASET SIZES
# ------------------------------------------------------------

print(
    "\nTraining admissions:",
    len(X_train)
)

print(
    "Testing admissions:",
    len(X_test)
)

print(
    "Training patients:",
    train_subjects.nunique()
)

print(
    "Testing patients:",
    test_subjects.nunique()
)


# ------------------------------------------------------------
# 5. CHECK CLASS DISTRIBUTION
# ------------------------------------------------------------

print("\nTraining target distribution:")

print(
    y_train.value_counts()
)

print("\nTraining positive percentage:")

print(
    y_train.mean() * 100
)


print("\nTesting target distribution:")

print(
    y_test.value_counts()
)

print("\nTesting positive percentage:")

print(
    y_test.mean() * 100
)


print("\nSTEP 25 COMPLETE")
# ============================================================
# STEP 26: MISSING VALUE IMPUTATION
# ============================================================

print("\n" + "=" * 50)
print("STEP 26: MISSING VALUE IMPUTATION")
print("=" * 50)

from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 1. IDENTIFY FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "gender",
    "admission_type",
    "admission_location"
]

numerical_features = [
    col
    for col in X_train.columns
    if col not in categorical_features
]


print(
    "\nNumerical features:",
    len(numerical_features)
)

print(
    "Categorical features:",
    len(categorical_features)
)


# ------------------------------------------------------------
# 2. NUMERICAL FEATURES -> MEDIAN IMPUTATION
# Fit ONLY on training data
# ------------------------------------------------------------

median_imputer = SimpleImputer(
    strategy="median"
)

X_train[numerical_features] = (
    median_imputer.fit_transform(
        X_train[numerical_features]
    )
)

X_test[numerical_features] = (
    median_imputer.transform(
        X_test[numerical_features]
    )
)


# ------------------------------------------------------------
# 3. CATEGORICAL FEATURES -> MODE IMPUTATION
# ------------------------------------------------------------

categorical_imputer = SimpleImputer(
    strategy="most_frequent"
)

X_train[categorical_features] = (
    categorical_imputer.fit_transform(
        X_train[categorical_features]
    )
)

X_test[categorical_features] = (
    categorical_imputer.transform(
        X_test[categorical_features]
    )
)


# ------------------------------------------------------------
# 4. VERIFY
# ------------------------------------------------------------

print(
    "\nMissing values remaining in training:",
    X_train.isna().sum().sum()
)

print(
    "Missing values remaining in testing:",
    X_test.isna().sum().sum()
)


print("\nSTEP 26 COMPLETE")
# ============================================================
# STEP 27: ONE-HOT ENCODE CATEGORICAL FEATURES
# ============================================================

print("\n" + "=" * 50)
print("STEP 27: ENCODE CATEGORICAL FEATURES")
print("=" * 50)

from sklearn.preprocessing import OneHotEncoder


# ------------------------------------------------------------
# 1. CREATE ENCODER
# ------------------------------------------------------------

encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop="first",
    sparse_output=False
)


# ------------------------------------------------------------
# 2. FIT ONLY ON TRAINING DATA
# ------------------------------------------------------------

train_encoded = encoder.fit_transform(
    X_train[categorical_features]
)

test_encoded = encoder.transform(
    X_test[categorical_features]
)


# ------------------------------------------------------------
# 3. GET NEW COLUMN NAMES
# ------------------------------------------------------------

encoded_columns = (
    encoder.get_feature_names_out(
        categorical_features
    )
)


# ------------------------------------------------------------
# 4. CONVERT TO DATAFRAMES
# ------------------------------------------------------------

train_encoded_df = pd.DataFrame(
    train_encoded,
    columns=encoded_columns,
    index=X_train.index
)

test_encoded_df = pd.DataFrame(
    test_encoded,
    columns=encoded_columns,
    index=X_test.index
)


# ------------------------------------------------------------
# 5. REMOVE ORIGINAL TEXT COLUMNS
# ------------------------------------------------------------

X_train = X_train.drop(
    columns=categorical_features
)

X_test = X_test.drop(
    columns=categorical_features
)


# ------------------------------------------------------------
# 6. ADD ENCODED COLUMNS
# ------------------------------------------------------------

X_train = pd.concat(
    [
        X_train,
        train_encoded_df
    ],
    axis=1
)

X_test = pd.concat(
    [
        X_test,
        test_encoded_df
    ],
    axis=1
)


# ------------------------------------------------------------
# 7. VERIFY
# ------------------------------------------------------------

print(
    "\nEncoded categorical columns:",
    len(encoded_columns)
)

print(
    "Training shape:",
    X_train.shape
)

print(
    "Testing shape:",
    X_test.shape
)

print(
    "\nMissing training values:",
    X_train.isna().sum().sum()
)

print(
    "Missing testing values:",
    X_test.isna().sum().sum()
)

print(
    "\nAll columns numeric:",
    all(
        pd.api.types.is_numeric_dtype(
            X_train[col]
        )
        for col in X_train.columns
    )
)


print("\nSTEP 27 COMPLETE")
# ============================================================
# STEP 28: STANDARDIZE FEATURES
# ============================================================

print("\n" + "=" * 50)
print("STEP 28: STANDARDIZE FEATURES")
print("=" * 50)

from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# 1. CREATE SCALER
# ------------------------------------------------------------

scaler = StandardScaler()


# ------------------------------------------------------------
# 2. FIT ON TRAINING DATA ONLY
# ------------------------------------------------------------

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)


# ------------------------------------------------------------
# 3. APPLY SAME SCALER TO TEST DATA
# ------------------------------------------------------------

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)


# ------------------------------------------------------------
# 4. VERIFY
# ------------------------------------------------------------

print(
    "\nTraining shape:",
    X_train_scaled.shape
)

print(
    "Testing shape:",
    X_test_scaled.shape
)


print(
    "\nAverage training feature mean:",
    X_train_scaled.mean().mean()
)

print(
    "Average training feature SD:",
    X_train_scaled.std(ddof=0).mean()
)


print(
    "\nMissing training values:",
    X_train_scaled.isna().sum().sum()
)

print(
    "Missing testing values:",
    X_test_scaled.isna().sum().sum()
)


print("\nSTEP 28 COMPLETE")
# ============================================================
# VISUALIZATION BEFORE VIF
# Missingness + Correlation
# ============================================================
# ============================================================
# PRE-VIF DATA VISUALIZATION
# Missingness + Correlation
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("\n" + "=" * 60)
print("PRE-VIF DATA VISUALIZATION")
print("=" * 60)

# ============================================================
# FIGURE 1: MISSING VALUE HEATMAP
# BEFORE IMPUTATION
# ============================================================

# clean_data is the dataset BEFORE Step 26 imputation.
# Remove IDs and target because they are not predictors.

missing_features = clean_data.drop(
    columns=["subject_id", "hadm_id", "icu_transfer"],
    errors="ignore"
)

# Sort features by missingness
missing_percent = (
    missing_features
    .isna()
    .mean()
    .mul(100)
    .sort_values(ascending=False)
)

# Keep features that actually contain missing values
missing_percent = missing_percent[
    missing_percent > 0
]

plt.figure(figsize=(10, 8))

plt.imshow(
    missing_percent.values.reshape(-1, 1),
    aspect="auto"
)

plt.yticks(
    np.arange(len(missing_percent)),
    missing_percent.index,
    fontsize=7
)

plt.xticks(
    [0],
    ["Missing (%)"]
)

plt.colorbar(
    label="Missing Data (%)"
)

plt.title(
    "Missingness of Candidate Predictors Before Imputation"
)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR / "PREVIF_01_missingness_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 2: CORRELATION HEATMAP BEFORE VIF
# ============================================================

# X_train_scaled is already:
# - imputed
# - encoded
# - standardized
# and represents the 77 predictors entering VIF.

corr_matrix = X_train_scaled.corr(
    method="pearson"
)

plt.figure(figsize=(16, 14))

plt.imshow(
    corr_matrix,
    aspect="auto",
    vmin=-1,
    vmax=1
)

plt.colorbar(
    label="Pearson Correlation Coefficient"
)

plt.xticks(
    np.arange(len(corr_matrix.columns)),
    corr_matrix.columns,
    rotation=90,
    fontsize=5
)

plt.yticks(
    np.arange(len(corr_matrix.columns)),
    corr_matrix.columns,
    fontsize=5
)

plt.title(
    "Correlation Matrix of Predictors Before VIF"
)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR / "PREVIF_02_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 3: TOP HIGHLY CORRELATED FEATURE PAIRS
# ============================================================

# This makes the correlation result easier to interpret
# than the full 77 x 77 heatmap.

upper_triangle = corr_matrix.where(
    np.triu(
        np.ones(corr_matrix.shape),
        k=1
    ).astype(bool)
)

correlation_pairs = (
    upper_triangle
    .stack()
    .reset_index()
)

correlation_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

correlation_pairs[
    "Absolute_Correlation"
] = (
    correlation_pairs[
        "Correlation"
    ].abs()
)

correlation_pairs = (
    correlation_pairs
    .sort_values(
        "Absolute_Correlation",
        ascending=False
    )
)

top_pairs = correlation_pairs.head(20)

print(
    "\nTOP 20 CORRELATED FEATURE PAIRS BEFORE VIF:\n"
)

print(
    top_pairs.to_string(
        index=False
    )
)


# Plot top correlations

plot_pairs = top_pairs.copy()

plot_pairs["Pair"] = (
    plot_pairs["Feature_1"]
    + "\nvs\n"
    + plot_pairs["Feature_2"]
)

plot_pairs = plot_pairs.sort_values(
    "Absolute_Correlation"
)

plt.figure(figsize=(10, 9))

plt.barh(
    plot_pairs["Pair"],
    plot_pairs["Absolute_Correlation"]
)

plt.xlabel(
    "Absolute Pearson Correlation"
)

plt.ylabel(
    "Feature Pair"
)

plt.title(
    "Top 20 Correlated Predictor Pairs Before VIF"
)

plt.xlim(0, 1)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR / "PREVIF_03_top_correlated_pairs.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\nPRE-VIF VISUALIZATION COMPLETE")
# ============================================================
# STEP 29: VARIANCE INFLATION FACTOR (VIF)
# ============================================================

print("\n" + "=" * 50)
print("STEP 29: KIDNEY-COHORT VIF FEATURE SELECTION")
print("=" * 50)

from statsmodels.stats.outliers_influence import variance_inflation_factor


# ------------------------------------------------------------
# 1. START WITH ALL STANDARDIZED TRAINING FEATURES
# ------------------------------------------------------------

X_vif = X_train_scaled.copy()

print(
    "\nStarting number of predictors:",
    X_vif.shape[1]
)


# ------------------------------------------------------------
# 2. FUNCTION TO CALCULATE VIF
# ------------------------------------------------------------

def calculate_vif(data):

    vif_table = pd.DataFrame()

    vif_table["feature"] = (
        data.columns
    )

    vif_table["VIF"] = [

        variance_inflation_factor(
            data.values,
            i
        )

        for i in range(
            data.shape[1]
        )

    ]

    return (
        vif_table
        .sort_values(
            "VIF",
            ascending=False
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 3. ITERATIVELY REMOVE HIGH-VIF FEATURES
# ------------------------------------------------------------

VIF_THRESHOLD = 10.0

removed_vif_features = []


while True:

    print(
        "\nCalculating VIF for",
        X_vif.shape[1],
        "features..."
    )

    vif_table = calculate_vif(
        X_vif
    )

    max_vif = (
        vif_table.iloc[0]["VIF"]
    )

    max_feature = (
        vif_table.iloc[0]["feature"]
    )


    print(
        "Highest VIF:",
        round(
            max_vif,
            3
        ),
        "| Feature:",
        max_feature
    )


    # Stop when all VIF values are acceptable
    if (
        max_vif <= VIF_THRESHOLD
        or
        X_vif.shape[1] <= 1
    ):

        break


    # Remove feature with highest VIF
    removed_vif_features.append(
        max_feature
    )

    X_vif = X_vif.drop(
        columns=[
            max_feature
        ]
    )


# ------------------------------------------------------------
# 4. FINAL KIDNEY-COHORT FEATURE SET
# ------------------------------------------------------------

vif_selected_features = (
    X_vif.columns.tolist()
)


print("\n" + "-" * 50)

print(
    "Original features:",
    X_train_scaled.shape[1]
)

print(
    "Features removed by VIF:",
    len(
        removed_vif_features
    )
)

print(
    "Features remaining:",
    len(
        vif_selected_features
    )
)


print(
    "\nFeatures removed:"
)

for feature in removed_vif_features:

    print(
        " -",
        feature
    )


# ------------------------------------------------------------
# 5. FINAL VIF TABLE
# ------------------------------------------------------------

print(
    "\nFinal VIF table:\n"
)

print(
    vif_table.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 6. APPLY SELECTED FEATURES TO TRAINING DATA
# ------------------------------------------------------------

X_train_vif = (
    X_train_scaled[
        vif_selected_features
    ]
    .copy()
)


# ------------------------------------------------------------
# 7. APPLY EXACT SAME FEATURES TO TEST DATA
# ------------------------------------------------------------

X_test_vif = (
    X_test_scaled[
        vif_selected_features
    ]
    .copy()
)


print(
    "\nTraining shape after VIF:",
    X_train_vif.shape
)

print(
    "Testing shape after VIF:",
    X_test_vif.shape
)


# ------------------------------------------------------------
# 8. SAVE VIF RESULTS
# These are outputs, NOT caches being loaded
# ------------------------------------------------------------

vif_table.to_csv(

    PROJECT_DIR
    / "kidney_final_vif_table.csv",

    index=False

)


with open(

    PROJECT_DIR
    / "kidney_vif_selected_features.txt",

    "w"

) as file:

    for feature in vif_selected_features:

        file.write(
            feature + "\n"
        )


with open(

    PROJECT_DIR
    / "kidney_vif_removed_features.txt",

    "w"

) as file:

    for feature in removed_vif_features:

        file.write(
            feature + "\n"
        )


print(
    "\nKidney-specific VIF results saved."
)

print("\nSTEP 29 COMPLETE")
# ============================================================
# STEP 30: RECURSIVE FEATURE ELIMINATION (RFE)
# XGBOOST-BASED FEATURE SELECTION
# ============================================================

print("\n" + "=" * 50)
print("STEP 30: KIDNEY-COHORT RFE FEATURE SELECTION")
print("=" * 50)

from sklearn.feature_selection import RFE
from xgboost import XGBClassifier


# ------------------------------------------------------------
# 1. CREATE XGBOOST ESTIMATOR FOR FEATURE SELECTION
# ------------------------------------------------------------

rfe_estimator = XGBClassifier(

    n_estimators=150,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.8,

    colsample_bytree=0.8,

    objective="binary:logistic",

    eval_metric="logloss",

    random_state=42,

    n_jobs=-1

)


# ------------------------------------------------------------
# 2. CREATE RFE
# ------------------------------------------------------------

rfe = RFE(

    estimator=rfe_estimator,

    # Keep final 20 predictors
    n_features_to_select=20,

    # Remove 2 features per iteration
    step=2,

    verbose=1

)


# ------------------------------------------------------------
# 3. FIT RFE USING TRAINING DATA ONLY
# ------------------------------------------------------------

print(
    "\nStarting XGBoost RFE with",
    X_train_vif.shape[1],
    "features..."
)


rfe.fit(
    X_train_vif,
    y_train
)


# ------------------------------------------------------------
# 4. GET FINAL SELECTED FEATURES
# ------------------------------------------------------------

rfe_selected_features = (

    X_train_vif.columns[
        rfe.support_
    ]

    .tolist()

)


print(
    "\nNumber of selected features:",
    len(
        rfe_selected_features
    )
)


print(
    "\nSelected features:\n"
)


for i, feature in enumerate(
    rfe_selected_features,
    start=1
):

    print(
        f"{i:2d}. {feature}"
    )


# ------------------------------------------------------------
# 5. CREATE COMPLETE RFE RANKING TABLE
# ------------------------------------------------------------

rfe_ranking = pd.DataFrame({

    "feature":
        X_train_vif.columns,

    "RFE_rank":
        rfe.ranking_,

    "selected":
        rfe.support_

}).sort_values(

    [
        "selected",
        "RFE_rank"
    ],

    ascending=[
        False,
        True
    ]

)


print(
    "\nFull RFE ranking:\n"
)


print(
    rfe_ranking.to_string(
        index=False
    )
)


# ------------------------------------------------------------
# 6. APPLY FINAL FEATURES TO TRAINING DATA
# ------------------------------------------------------------

X_train_rfe = X_train_vif[
    rfe_selected_features
].copy()


# ------------------------------------------------------------
# 7. APPLY EXACT SAME FEATURES TO TEST DATA
# ------------------------------------------------------------

X_test_rfe = X_test_vif[
    rfe_selected_features
].copy()


print(
    "\nTraining shape after RFE:",
    X_train_rfe.shape
)

print(
    "Testing shape after RFE:",
    X_test_rfe.shape
)


print(
    "\nTrain/test columns identical:",
    X_train_rfe.columns.tolist()
    ==
    X_test_rfe.columns.tolist()
)


# ------------------------------------------------------------
# 8. SAVE COMPLETE RFE RANKING
# ------------------------------------------------------------

rfe_ranking.to_csv(

    PROJECT_DIR /
    "kidney_rfe_feature_ranking.csv",

    index=False

)


# ------------------------------------------------------------
# 9. SAVE FINAL 20 FEATURES
# This is an OUTPUT of the new RFE run.
# It is NOT being loaded as a cache.
# ------------------------------------------------------------

RFE_FEATURE_FILE = (

    PROJECT_DIR /
    "kidney_rfe_selected_features.txt"

)


with open(
    RFE_FEATURE_FILE,
    "w"
) as file:

    for feature in rfe_selected_features:

        file.write(
            feature + "\n"
        )


print(
    "\nKidney-specific RFE features saved to:"
)

print(
    RFE_FEATURE_FILE
)


print("\nSTEP 30 COMPLETE")
# ============================================================
# SAVE FINAL MODEL-READY DATASETS
# ============================================================

train_final = X_train_rfe.copy()
train_final["icu_transfer"] = y_train.values

test_final = X_test_rfe.copy()
test_final["icu_transfer"] = y_test.values


train_final.to_csv(
    PROJECT_DIR / "kidney_final_train_dataset.csv",
    index=False
)

test_final.to_csv(
    PROJECT_DIR / "kidney_final_test_dataset.csv",
    index=False
)


print("\nFinal cleaned datasets saved:")
print("kidney_final_train_dataset.csv")
print("kidney_final_test_dataset.csv")
# ============================================================
# STEP 31: KIDNEY-COHORT CLASS IMBALANCE
# ============================================================

print("\n" + "=" * 50)
print("STEP 31: KIDNEY-COHORT CLASS IMBALANCE")
print("=" * 50)


# ------------------------------------------------------------
# 1. COUNT POSITIVE AND NEGATIVE TRAINING CASES
# ------------------------------------------------------------

negative_count = (
    y_train == 0
).sum()

positive_count = (
    y_train == 1
).sum()


print(
    "\nKidney patients without ICU transfer:",
    negative_count
)

print(
    "Kidney patients with ICU transfer:",
    positive_count
)


# ------------------------------------------------------------
# 2. CALCULATE CLASS IMBALANCE
# ------------------------------------------------------------

if positive_count == 0:

    raise ValueError(
        "No positive ICU-transfer cases found "
        "in the kidney training cohort."
    )


scale_pos_weight = (
    negative_count
    /
    positive_count
)


positive_percentage = (
    positive_count
    /
    len(y_train)
    * 100
)


print(
    "\nPositive ICU-transfer percentage:",
    round(
        positive_percentage,
        3
    ),
    "%"
)


print(
    "Negative : Positive ratio:",
    round(
        scale_pos_weight,
        2
    ),
    ": 1"
)


print(
    "\nXGBoost scale_pos_weight:",
    round(
        scale_pos_weight,
        4
    )
)


# ------------------------------------------------------------
# 3. MODEL SETTINGS TO USE LATER
# ------------------------------------------------------------

print(
    "\nFor Logistic Regression / Random Forest:"
)

print(
    'Use class_weight="balanced"'
)


print(
    "\nFor XGBoost:"
)

print(
    "Use scale_pos_weight =",
    round(
        scale_pos_weight,
        4
    )
)


# ------------------------------------------------------------
# 4. VERIFY TEST SET REMAINS UNCHANGED
# ------------------------------------------------------------

print(
    "\nTraining observations:",
    len(y_train)
)

print(
    "Testing observations:",
    len(y_test)
)


print(
    "\nNo oversampling applied."
)

print(
    "Test-set distribution remains unchanged."
)


# ------------------------------------------------------------
# 5. SAVE CLASS-BALANCE INFORMATION
# ------------------------------------------------------------

class_balance_summary = pd.DataFrame({

    "metric": [
        "negative_training_cases",
        "positive_training_cases",
        "positive_percentage",
        "negative_to_positive_ratio"
    ],

    "value": [
        negative_count,
        positive_count,
        positive_percentage,
        scale_pos_weight
    ]

})


class_balance_summary.to_csv(

    PROJECT_DIR /
    "kidney_class_balance_summary.csv",

    index=False

)


print(
    "\nKidney class-balance summary saved."
)


print("\nSTEP 31 COMPLETE")
# ============================================================
# STEP 32: EDA OF CLEANED / PROCESSED KIDNEY-DISEASE DATASET
# ============================================================

print("\n" + "=" * 60)
print("STEP 32: EDA OF CLEANED KIDNEY-DISEASE DATASET")
print("=" * 60)

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


# ============================================================
# FIGURE 1: AGE DISTRIBUTION
# ============================================================

age_data = model_cohort["age"].dropna()

plt.figure(figsize=(8, 5))

plt.hist(
    age_data,
    bins=30,
    edgecolor="black"
)

plt.xlabel("Age (years)")
plt.ylabel("Number of Admissions")
plt.title("Age Distribution of Kidney-Disease Cohort")

plt.tight_layout()

plt.savefig(
    DATA_VIS_DIR / "EDA_01_age_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 2: SEX DISTRIBUTION
# ============================================================

gender_counts = (
    model_cohort["gender"]
    .value_counts()
)

plt.figure(figsize=(6, 5))

bars = plt.bar(
    gender_counts.index,
    gender_counts.values
)

plt.xlabel("Sex")
plt.ylabel("Number of Admissions")
plt.title("Sex Distribution of Kidney-Disease Cohort")

for bar, value in zip(
    bars,
    gender_counts.values
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{value:,}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_02_sex_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 3: AKI / CKD COHORT COMPOSITION
# ============================================================

kidney_counts = (
    model_cohort["kidney_group"]
    .value_counts()
)

plt.figure(figsize=(8, 5))

bars = plt.bar(
    kidney_counts.index,
    kidney_counts.values
)

plt.xlabel("Kidney Disease Group")
plt.ylabel("Number of Admissions")
plt.title("Composition of Kidney-Disease Cohort")

for bar, value in zip(
    bars,
    kidney_counts.values
):

    percentage = (
        value
        / kidney_counts.sum()
        * 100
    )

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{value:,}\n({percentage:.1f}%)",
        ha="center",
        va="bottom"
    )

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_03_kidney_group_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 4: ICU TRANSFER OUTCOME
# ============================================================

outcome_counts = (
    model_cohort["icu_transfer"]
    .value_counts()
    .sort_index()
)

outcome_values = [
    outcome_counts.get(0, 0),
    outcome_counts.get(1, 0)
]

outcome_labels = [
    "No ICU Transfer",
    "ICU Transfer 24-72 h"
]

plt.figure(figsize=(7, 5))

bars = plt.bar(
    outcome_labels,
    outcome_values
)

total = sum(outcome_values)

for bar, value in zip(
    bars,
    outcome_values
):

    percentage = (
        value
        / total
        * 100
    )

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{value:,}\n({percentage:.2f}%)",
        ha="center",
        va="bottom"
    )

plt.ylabel("Number of Admissions")
plt.title("ICU Transfer Outcome")

plt.ylim(
    0,
    max(outcome_values) * 1.15
)

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_04_icu_outcome.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 5: AGE BY ICU OUTCOME
# ============================================================

no_icu_age = (
    model_cohort.loc[
        model_cohort["icu_transfer"] == 0,
        "age"
    ]
    .dropna()
)

icu_age = (
    model_cohort.loc[
        model_cohort["icu_transfer"] == 1,
        "age"
    ]
    .dropna()
)

plt.figure(figsize=(7, 5))

plt.boxplot(
    [
        no_icu_age,
        icu_age
    ],
    tick_labels=[
        "No ICU Transfer",
        "ICU Transfer"
    ],
    showfliers=False
)

plt.ylabel("Age (years)")
plt.title("Age by ICU Transfer Outcome")

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_05_age_by_outcome.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 6: ADMISSION TYPE
# ============================================================

admission_counts = (
    model_cohort["admission_type"]
    .value_counts()
    .sort_values()
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    admission_counts.index,
    admission_counts.values
)

plt.xlabel("Number of Admissions")
plt.ylabel("Admission Type")
plt.title("Admission Types in Kidney-Disease Cohort")

for bar, value in zip(
    bars,
    admission_counts.values
):

    plt.text(
        bar.get_width(),
        bar.get_y() + bar.get_height()/2,
        f" {value:,}",
        va="center"
    )

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_06_admission_type.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 7: CREATININE DISTRIBUTION
# ============================================================

creatinine = (
    reduced_data[
        "creatinine_latest_value"
    ]
    .dropna()
)

# 99th percentile restriction is FOR VISUALIZATION ONLY
creatinine_upper = creatinine.quantile(0.99)

creatinine_plot = creatinine[
    creatinine <= creatinine_upper
]

plt.figure(figsize=(8, 5))

plt.hist(
    creatinine_plot,
    bins=40,
    edgecolor="black"
)

plt.xlabel("Latest Creatinine (mg/dL)")
plt.ylabel("Number of Admissions")
plt.title("Creatinine Distribution")

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_07_creatinine_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 8: BUN DISTRIBUTION
# ============================================================

bun = (
    reduced_data[
        "bun_latest_value"
    ]
    .dropna()
)

bun_upper = bun.quantile(0.99)

bun_plot = bun[
    bun <= bun_upper
]

plt.figure(figsize=(8, 5))

plt.hist(
    bun_plot,
    bins=40,
    edgecolor="black"
)

plt.xlabel("Latest BUN (mg/dL)")
plt.ylabel("Number of Admissions")
plt.title("BUN Distribution")

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_08_bun_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 9: POTASSIUM DISTRIBUTION
# ============================================================

potassium = (
    reduced_data[
        "potassium_latest_value"
    ]
    .dropna()
)

plt.figure(figsize=(8, 5))

plt.hist(
    potassium,
    bins=40,
    edgecolor="black"
)

plt.xlabel("Latest Potassium (mEq/L)")
plt.ylabel("Number of Admissions")
plt.title("Potassium Distribution")

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_09_potassium_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 10: CREATININE BY ICU OUTCOME
# ============================================================

creatinine_outcome = model_data[
    [
        "creatinine_latest_value",
        "icu_transfer"
    ]
].dropna()

upper = (
    creatinine_outcome[
        "creatinine_latest_value"
    ]
    .quantile(0.99)
)

creatinine_outcome = creatinine_outcome[
    creatinine_outcome[
        "creatinine_latest_value"
    ] <= upper
]

no_icu_creatinine = (
    creatinine_outcome.loc[
        creatinine_outcome["icu_transfer"] == 0,
        "creatinine_latest_value"
    ]
)

icu_creatinine = (
    creatinine_outcome.loc[
        creatinine_outcome["icu_transfer"] == 1,
        "creatinine_latest_value"
    ]
)

plt.figure(figsize=(7, 5))

plt.boxplot(
    [
        no_icu_creatinine,
        icu_creatinine
    ],
    tick_labels=[
        "No ICU Transfer",
        "ICU Transfer"
    ],
    showfliers=False
)

plt.ylabel("Latest Creatinine (mg/dL)")
plt.title("Creatinine by ICU Transfer Outcome")

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_10_creatinine_by_outcome.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 11: BUN BY ICU OUTCOME
# ============================================================

bun_outcome = model_data[
    [
        "bun_latest_value",
        "icu_transfer"
    ]
].dropna()

upper = (
    bun_outcome[
        "bun_latest_value"
    ]
    .quantile(0.99)
)

bun_outcome = bun_outcome[
    bun_outcome[
        "bun_latest_value"
    ] <= upper
]

no_icu_bun = (
    bun_outcome.loc[
        bun_outcome["icu_transfer"] == 0,
        "bun_latest_value"
    ]
)

icu_bun = (
    bun_outcome.loc[
        bun_outcome["icu_transfer"] == 1,
        "bun_latest_value"
    ]
)

plt.figure(figsize=(7, 5))

plt.boxplot(
    [
        no_icu_bun,
        icu_bun
    ],
    tick_labels=[
        "No ICU Transfer",
        "ICU Transfer"
    ],
    showfliers=False
)

plt.ylabel("Latest BUN (mg/dL)")
plt.title("BUN by ICU Transfer Outcome")

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR/ "EDA_11_bun_by_outcome.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 12: POTASSIUM BY ICU OUTCOME
# ============================================================

potassium_outcome = model_data[
    [
        "potassium_latest_value",
        "icu_transfer"
    ]
].dropna()

no_icu_potassium = (
    potassium_outcome.loc[
        potassium_outcome["icu_transfer"] == 0,
        "potassium_latest_value"
    ]
)

icu_potassium = (
    potassium_outcome.loc[
        potassium_outcome["icu_transfer"] == 1,
        "potassium_latest_value"
    ]
)

plt.figure(figsize=(7, 5))

plt.boxplot(
    [
        no_icu_potassium,
        icu_potassium
    ],
    tick_labels=[
        "No ICU Transfer",
        "ICU Transfer"
    ],
    showfliers=False
)

plt.ylabel("Latest Potassium (mEq/L)")
plt.title("Potassium by ICU Transfer Outcome")

plt.tight_layout()

plt.savefig(
     DATA_VIS_DIR / "EDA_12_potassium_by_outcome.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# PRINT EDA SUMMARY
# ============================================================

print("\nCLEANED DATASET SUMMARY")

print(
    "\nEligible admissions:",
    f"{len(model_cohort):,}"
)

print(
    "Unique patients:",
    f"{model_cohort['subject_id'].nunique():,}"
)

print("\nKidney groups:")

print(
    model_cohort[
        "kidney_group"
    ].value_counts()
)

print("\nICU outcome:")

print(
    model_cohort[
        "icu_transfer"
    ].value_counts()
)

print("\nSTEP 32 COMPLETE")
# ============================================================
# STEP 33: FEATURE-SELECTION VISUALIZATION
# ============================================================

print("\n" + "=" * 60)
print("STEP 33: FEATURE-SELECTION VISUALIZATION")
print("=" * 60)

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


# ============================================================
# FIGURE 1: COMPLETE FEATURE-REDUCTION PIPELINE
# ============================================================

stages = [
    "Engineered\nLab Features",
    "Retained Lab\nRepresentations",
    "Candidate Predictors\nAfter Missingness",
    "After One-Hot\nEncoding",
    "After VIF",
    "After RFE"
]

feature_counts = [
    252,
    112,
    61,
    77,
    60,
    20
]


plt.figure(figsize=(11, 5))

bars = plt.bar(
    stages,
    feature_counts
)

plt.ylabel("Number of Features / Predictors")

plt.title(
    "Feature-Reduction Pipeline"
)


for bar, value in zip(
    bars,
    feature_counts
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 4,
        str(value),
        ha="center",
        fontweight="bold"
    )


plt.ylim(
    0,
    285
)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR /
    "FS_01_feature_reduction_pipeline.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 2: VIF REDUCTION
# ============================================================

vif_stages = [
    "Before VIF",
    "After VIF"
]

vif_counts = [
    X_train_scaled.shape[1],
    X_train_vif.shape[1]
]


plt.figure(figsize=(6, 5))

bars = plt.bar(
    vif_stages,
    vif_counts
)

plt.ylabel("Number of Predictors")

plt.title(
    "Multicollinearity Reduction Using VIF"
)


for bar, value in zip(
    bars,
    vif_counts
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        str(value),
        ha="center",
        fontweight="bold"
    )


plt.ylim(
    0,
    max(vif_counts) + 15
)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR /
    "FS_02_vif_reduction.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 3: RFE REDUCTION
# ============================================================

rfe_stages = [
    "Before RFE",
    "After RFE"
]

rfe_counts = [
    X_train_vif.shape[1],
    X_train_rfe.shape[1]
]


plt.figure(figsize=(6, 5))

bars = plt.bar(
    rfe_stages,
    rfe_counts
)

plt.ylabel("Number of Predictors")

plt.title(
    "Recursive Feature Elimination"
)


for bar, value in zip(
    bars,
    rfe_counts
):

    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 1,
        str(value),
        ha="center",
        fontweight="bold"
    )


plt.ylim(
    0,
    max(rfe_counts) + 15
)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR /
    "FS_03_rfe_reduction.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 4: FINAL 20 SELECTED FEATURES
# ============================================================

pretty_features = {

    "admission_hour":
        "Admission hour",

    "anion_gap_latest_value":
        "Anion gap",

    "calcium_latest_value":
        "Calcium",

    "creatinine_latest_value":
        "Creatinine",

    "glucose_latest_value":
        "Glucose",

    "hematocrit_latest_value":
        "Hematocrit",

    "pt_latest_value":
        "Prothrombin time",

    "wbc_latest_value":
        "WBC",

    "inr_count":
        "INR measurement count",

    "phosphate_count":
        "Phosphate measurement count",

    "platelets_count":
        "Platelet measurement count",

    "bun_change":
        "BUN change",

    "wbc_change":
        "WBC change",

    "admission_type_DIRECT OBSERVATION":
        "Direct observation",

    "admission_type_ELECTIVE":
        "Elective admission",

    "admission_type_EU OBSERVATION":
        "EU observation",

    "admission_type_SURGICAL SAME DAY ADMISSION":
        "Same-day surgery",

    "admission_type_URGENT":
        "Urgent admission",

    "admission_location_PHYSICIAN REFERRAL":
        "Physician referral",

    "admission_location_TRANSFER FROM HOSPITAL":
        "Transfer from hospital"
}


final_feature_names = [

    pretty_features.get(
        feature,
        feature
    )

    for feature
    in rfe_selected_features
]


# RFE selected all 20 equally at the final stage,
# so show them as a ranked/list-style visualization rather
# than pretending these are model importance scores.

positions = np.arange(
    len(final_feature_names)
)


plt.figure(
    figsize=(10, 8)
)

plt.scatter(
    np.ones(
        len(final_feature_names)
    ),
    positions
)


for position, name in zip(
    positions,
    final_feature_names
):

    plt.text(
        1.02,
        position,
        name,
        va="center"
    )


plt.yticks(
    positions,
    [
        f"{i}"
        for i
        in range(
            1,
            len(final_feature_names) + 1
        )
    ]
)

plt.xticks([])

plt.ylabel(
    "Selected Feature Number"
)

plt.title(
    "Final 20 Features Selected by RFE"
)

plt.ylim(
    len(final_feature_names),
    -1
)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR /
    "FS_04_final_20_features.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# FIGURE 5: GENERAL VS KIDNEY FEATURE SELECTION
# ============================================================

stages_compare = [
    "After Encoding",
    "After VIF",
    "After RFE"
]

general_features = [
    71,
    56,
    20
]

kidney_features = [
    77,
    60,
    20
]


x = np.arange(
    len(stages_compare)
)

width = 0.35


plt.figure(
    figsize=(9, 5)
)


bars1 = plt.bar(
    x - width/2,
    general_features,
    width,
    label="General Cohort"
)


bars2 = plt.bar(
    x + width/2,
    kidney_features,
    width,
    label="Kidney Cohort"
)


plt.xticks(
    x,
    stages_compare
)

plt.ylabel(
    "Number of Predictors"
)

plt.title(
    "Feature Selection: General vs Kidney Cohort"
)

plt.legend()


for bars in [
    bars1,
    bars2
]:

    for bar in bars:

        value = int(
            bar.get_height()
        )

        plt.text(
            bar.get_x()
            + bar.get_width()/2,

            bar.get_height() + 1,

            str(value),

            ha="center"
        )


plt.ylim(
    0,
    90
)

plt.tight_layout()

plt.savefig(
    FEATURE_VIS_DIR/
    "FS_05_general_vs_kidney.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# CREATE FINAL FEATURE TABLE
# ============================================================

final_feature_table = pd.DataFrame({

    "Feature_Number":
        range(
            1,
            len(rfe_selected_features) + 1
        ),

    "Feature":
        rfe_selected_features,

    "Readable_Name":
        final_feature_names

})


print(
    "\nFINAL 20 SELECTED FEATURES:\n"
)

print(
    final_feature_table.to_string(
        index=False
    )
)


final_feature_table.to_csv(

    FEATURE_VIS_DIR /
    "final_20_kidney_features.csv",

    index=False
)


print(
    "\nFinal feature table saved."
)

print("\nSTEP 33 COMPLETE")
# ============================================================
# STEP 34: LOGISTIC REGRESSION BASELINE MODEL
# ============================================================

print("\n" + "=" * 60)
print("STEP 34: LOGISTIC REGRESSION BASELINE")
print("=" * 60)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve
)


# ============================================================
# 1. CREATE MODEL
# ============================================================

# class_weight="balanced" is used because ICU transfer
# is rare in the kidney cohort (~3%).

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)


# ============================================================
# 2. TRAIN MODEL
# ============================================================

print("\nTraining Logistic Regression...")

logistic_model.fit(
    X_train_rfe,
    y_train
)

print("Training complete.")


# ============================================================
# 3. PREDICT TEST SET
# ============================================================

y_pred = logistic_model.predict(
    X_test_rfe
)

y_prob = logistic_model.predict_proba(
    X_test_rfe
)[:, 1]


# ============================================================
# 4. CALCULATE PERFORMANCE METRICS
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

sensitivity = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

auroc = roc_auc_score(
    y_test,
    y_prob
)

auprc = average_precision_score(
    y_test,
    y_prob
)


# ============================================================
# 5. CONFUSION MATRIX + SPECIFICITY
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

tn, fp, fn, tp = cm.ravel()


specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0
)


# ============================================================
# 6. PRINT RESULTS
# ============================================================

print("\n" + "-" * 60)
print("LOGISTIC REGRESSION TEST PERFORMANCE")
print("-" * 60)

print(
    f"AUROC:       {auroc:.4f}"
)

print(
    f"AUPRC:       {auprc:.4f}"
)

print(
    f"Sensitivity: {sensitivity:.4f}"
)

print(
    f"Specificity: {specificity:.4f}"
)

print(
    f"Precision:   {precision:.4f}"
)

print(
    f"F1-score:    {f1:.4f}"
)

print(
    f"Accuracy:    {accuracy:.4f}"
)


print("\nConfusion Matrix:")

print(cm)


print("\nTN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)


print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4,
        zero_division=0
    )
)


# ============================================================
# 7. SAVE PERFORMANCE RESULTS
# ============================================================

results = pd.DataFrame({

    "Model": [
        "Logistic Regression"
    ],

    "AUROC": [
        auroc
    ],

    "AUPRC": [
        auprc
    ],

    "Sensitivity": [
        sensitivity
    ],

    "Specificity": [
        specificity
    ],

    "Precision": [
        precision
    ],

    "F1": [
        f1
    ],

    "Accuracy": [
        accuracy
    ]

})


results.to_csv(
    PROJECT_DIR /
    "logistic_regression_results.csv",
    index=False
)


print(
    "\nResults saved to:"
)

print(
    PROJECT_DIR /
    "logistic_regression_results.csv"
)


# ============================================================
# 8. CONFUSION MATRIX FIGURE
# ============================================================

plt.figure(
    figsize=(6, 5)
)

# ------------------------------------------------------------
# DISPLAY CONFUSION MATRIX
# Blue colour scheme
# ------------------------------------------------------------

plt.imshow(
    cm,
    cmap="Blues"
)

# Add colour bar
plt.colorbar(
    label="Number of Patients"
)


# ------------------------------------------------------------
# TITLE AND AXIS LABELS
# ------------------------------------------------------------

plt.title(
    "Logistic Regression Confusion Matrix",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel(
    "Predicted Class",
    fontsize=11
)

plt.ylabel(
    "Actual Class",
    fontsize=11
)


# ------------------------------------------------------------
# CLASS LABELS
# ------------------------------------------------------------

plt.xticks(
    [0, 1],
    [
        "No ICU",
        "ICU Transfer"
    ]
)

plt.yticks(
    [0, 1],
    [
        "No ICU",
        "ICU Transfer"
    ]
)


# ------------------------------------------------------------
# ADD VALUES INSIDE EACH CELL
# ------------------------------------------------------------

threshold = cm.max() / 2

for i in range(2):

    for j in range(2):

        # White text on dark cells
        # Black text on light cells
        text_color = (
            "white"
            if cm[i, j] > threshold
            else "black"
        )

        plt.text(
            j,
            i,
            f"{cm[i, j]:,}",
            ha="center",
            va="center",
            fontsize=13,
            fontweight="bold",
            color=text_color
        )


# ------------------------------------------------------------
# FINAL FORMATTING
# ------------------------------------------------------------

plt.subplots_adjust(
    left=0.18,
    right=0.88,
    bottom=0.20,
    top=0.88
)

plt.savefig(
    MODEL_VIS_DIR /
    "logistic_confusion_matrix_blue.png",
    dpi=300,
    bbox_inches="tight",
    pad_inches=0.3
)

plt.show()




# ============================================================
# 9. ROC CURVE
# ============================================================

fpr, tpr, roc_thresholds = roc_curve(
    y_test,
    y_prob
)


plt.figure(
    figsize=(6, 5)
)

plt.plot(
    fpr,
    tpr,
    label=f"Logistic Regression (AUROC = {auroc:.3f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve"
)

plt.legend(
    loc="lower right"
)

plt.tight_layout()

plt.savefig(
    MODEL_VIS_DIR /
    "logistic_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# ============================================================
# 10. PRECISION-RECALL CURVE
# ============================================================

pr_precision, pr_recall, pr_thresholds = (
    precision_recall_curve(
        y_test,
        y_prob
    )
)


baseline_prevalence = (
    y_test.mean()
)


plt.figure(
    figsize=(6, 5)
)

plt.plot(
    pr_recall,
    pr_precision,
    label=f"Logistic Regression (AUPRC = {auprc:.3f})"
)

plt.axhline(
    y=baseline_prevalence,
    linestyle="--",
    label=f"Baseline = {baseline_prevalence:.3f}"
)

plt.xlabel(
    "Recall (Sensitivity)"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Precision-Recall Curve"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    MODEL_VIS_DIR /
    "logistic_precision_recall_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
from sklearn.metrics import log_loss

bce_loss = log_loss(
    y_test,
    y_prob
)

print("\nSTEP 34 COMPLETE")